# Advanced Models
1. Try Gradient Boosting (e.g., XGBoost or LightGBM). 
2. Perform light hyperparameter tuning (depth, learning rate, n_estimators)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

X_train = pd.read_csv('X_train.csv')
X_val = pd.read_csv('X_val.csv')
X_test = pd.read_csv('X_test.csv')

Y_train = np.load('Y_train.npy')
Y_val = np.load('Y_val.npy')
Y_test = np.load('Y_test.npy')

# convert Y 2D arrays to 1D arrays
Y_train = pd.DataFrame(Y_train)[0].values
Y_val = pd.DataFrame(Y_val)[0].values
Y_test = pd.DataFrame(Y_test)[0].values

sklearn's Gradient Boosting

In [13]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, Y_train)

gb_pred = gb_model.predict(X_test)
print("R2:", r2_score(Y_test, gb_pred))

R2: 0.8037541997025232


XGBoost

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

xgb_model.fit(X_train, Y_train)

xgb_pred = xgb_model.predict(X_test)
print("R2: ", r2_score(Y_test, xgb_pred))

R2:  0.7861920932728163


In [ ]:
xgb_importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
xgb_importances.sort_values(ascending=False)

,0
PostalCode,0.264584
LivingArea,0.182895
BathroomsTotalInteger,0.122638
City,0.091767
ParkingTotal,0.062853
Longitude,0.042581
CountyOrParish,0.037537
MLSAreaMajor,0.035190
PoolPrivateYN_Unknown,0.032966
AttachedGarageYN_True,0.021419


LightGBM

In [ ]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    verbosity=-1 # remove warning outputs
)

lgb_model.fit(X_train, Y_train)

lgb_pred = lgb_model.predict(X_test)
print("R2: ", r2_score(Y_test, lgb_pred))

R2:  0.796188982827203


In [11]:
lgb_importances = pd.Series(lgb_model.feature_importances_, index=X_train.columns)
lgb_importances.sort_values(ascending=False)

LivingArea                  104
PostalCode                   96
MLSAreaMajor                 76
City                         70
BathroomsTotalInteger        48
YearBuilt                    42
Longitude                    38
Latitude                     36
LotSizeSquareFeet            31
HighSchoolDistrict           27
AssociationFee               26
CountyOrParish               21
BedroomsTotal                 9
PoolPrivateYN_Unknown         7
ParkingTotal                  6
GarageSpaces                  3
AttachedGarageYN_True         3
PoolPrivateYN_True            2
AttachedGarageYN_Unknown      1
ViewYN_Unknown                1
Stories                       0
ViewYN_True                   0
dtype: int32

Tune hyperparameters using RandomizedSearch

In [16]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform # uniform generates random floats

param_dist = {
    'max_depth': randint(3, 12), # how complex/deep each tree is
    'learning_rate': uniform(0.01, 0.3), # level of correction to make
    'n_estimators': randint(100, 1000) # how many trees
}

xgb_rand_search = RandomizedSearchCV(
    xgb_model,
    param_dist,
    n_iter=15,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
)

xgb_rand_search.fit(X_val, Y_val)
print("XGBoost best hyperparameters: ", xgb_rand_search.best_params_)

lgb_rand_search = RandomizedSearchCV(
    lgb_model,
    param_dist,
    n_iter=15,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=42,
)

lgb_rand_search.fit(X_val, Y_val)
print("LightGBM best hyperparameters: ", lgb_rand_search.best_params_)

XGBoost best hyperparameters:  {'learning_rate': np.float64(0.19355586841671385), 'max_depth': 5, 'n_estimators': 975}
LightGBM best hyperparameters:  {'learning_rate': np.float64(0.027425083650459835), 'max_depth': 10, 'n_estimators': 472}


Test models again with these optimal hyperparameters.

In [18]:
xgb_model_2 = xgb.XGBRegressor(
    n_estimators=975,
    learning_rate=0.1936,
    max_depth=5,
    random_state=42
)

xgb_model_2.fit(X_train, Y_train)

xgb_pred_2 = xgb_model_2.predict(X_test)
print("XGBoost R2 with new hyperparameters: ", r2_score(Y_test, xgb_pred_2))

lgb_model_2 = lgb.LGBMRegressor(
    n_estimators=472,
    learning_rate=0.0274,
    max_depth=10,
    random_state=42,
    verbosity=-1 # remove warning outputs
)

lgb_model_2.fit(X_train, Y_train)

lgb_pred_2 = lgb_model_2.predict(X_test)
print("LightGBM R2 with new hyperparameters: ", r2_score(Y_test, lgb_pred_2))

XGBoost R2 with new hyperparameters:  0.8163953576868576
LightGBM R2 with new hyperparameters:  0.815936215964875


We can see that the R2 score has improved.

In [21]:
print("XGBoost:")
print("Before: ", r2_score(Y_test, xgb_pred))
print("After: ", r2_score(Y_test, xgb_pred_2))
print("LightGBM:")
print("Before: ", r2_score(Y_test, lgb_pred))
print("After: ", r2_score(Y_test, lgb_pred_2))

XGBoost:
Before:  0.7861920932728163
After:  0.8163953576868576
LightGBM:
Before:  0.796188982827203
After:  0.815936215964875
